In [ ]:
# Install IMDLib
!pip -q install imdlib
# Install DFA5
!pip install cdsapi xarray netCDF4 numpy pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 81.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 63.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install "cdsapi>=0.7.7"

In [ ]:
# Re-install cdsapi after session termination if needed
!pip -q install cdsapi

In [ ]:
#this downloads IMD dataset
import os
import imdlib as imd

# ==========================================
# CONFIGURATION
# ==========================================

START_YEAR = 1986
END_YEAR   = 2025

# Change this path if required
BASE_DIR = "/content/drive/MyDrive/isro_climate_dataset"

os.makedirs(BASE_DIR, exist_ok=True)

# ==========================================
# DOWNLOAD IMD DATA
# ==========================================

VARIABLES = {
    "rain": "Rainfall",
    "TMax": "Maxtemp",
    "TMin": "Mintemp"
}

for imd_name, folder_name in VARIABLES.items():

    print("=" * 60)
    print(f"Downloading {folder_name}")
    print("=" * 60)

    save_dir = os.path.join(BASE_DIR, folder_name)
    os.makedirs(save_dir, exist_ok=True)

    imd.get_data(
        imd_name,
        START_YEAR,
        END_YEAR,
        fn_format="yearwise",
        file_dir=save_dir
    )

print("\n✅ All IMD datasets downloaded successfully!")

In [ ]:
%%writefile ~/.cdsapirc
url: https://cds.climate.copernicus.eu/api
key: 738cea5c-e52d-461b-8c8b-4d4ba6da3fa6

In [ ]:
#this downloads ERA5 dataset
import os
import cdsapi
import time
import calendar
from tqdm.auto import tqdm

# --------------------------------------------------
# SAVE LOCATION
# --------------------------------------------------

BASE = "/content/drive/MyDrive/isro_climate_dataset"

SINGLE = os.path.join(BASE, "ERA5", "SingleLevels")
PRESSURE = os.path.join(BASE, "ERA5", "PressureLevels")

os.makedirs(SINGLE, exist_ok=True)
os.makedirs(PRESSURE, exist_ok=True)

client = cdsapi.Client()

YEARS = list(range(1986, 2026))
MONTHS = [f"{i:02d}" for i in range(1, 13)]

# Changed to 00:00 UTC as recommended
TIME = ["00:00"]

# India Bounding Box
AREA = [
    38.5,   # North
    68.0,   # West
    6.0,    # South
    98.5    # East
]

def download_era5_data(
    dataset_name,
    variables,
    pressure_level,
    year,
    output_file,
    max_retries=5,
    delay=5
):
    """Downloads ERA5 data with retry and resume support."""

    if os.path.exists(output_file):
        print(f"  ✅ File already exists: {output_file}. Skipping download.")
        return

    print(f"  ⬇️ Downloading {dataset_name} for {year}...")

    for attempt in range(max_retries):
        try:
            request_payload = {
                "product_type": "reanalysis",
                "variable": variables,
                "year": str(year),
                "month": MONTHS,
                "day": [], # Will be populated per month
                "time": TIME,
                "area": AREA,
                "data_format": "netcdf"
            }

            if pressure_level: # Only add if specific to pressure levels
                 request_payload["pressure_level"] = pressure_level

            # Generate correct days for each month for the current year
            all_days = []
            for month_num_str in MONTHS:
                month_num = int(month_num_str)
                num_days = calendar.monthrange(year, month_num)[1]
                all_days.extend([f"{d:02d}" for d in range(1, num_days + 1)])
            request_payload["day"] = sorted(list(set(all_days))) # Use set to remove duplicates, then sort

            client.retrieve(
                dataset_name,
                request_payload,
                output_file
            )
            print(f"  ✅ Successfully downloaded {os.path.basename(output_file)}.")
            return

        except Exception as e:
            print(f"  ❌ Download failed for {os.path.basename(output_file)} (Attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in {delay} seconds...")
                time.sleep(delay)
                delay *= 2 # Exponential backoff
            else:
                print(f"  🛑 Max retries reached for {os.path.basename(output_file)}. Skipping this file.")
                return # Exit function if max retries reached


print("Starting ERA5 data download...")

for year in tqdm(YEARS, desc="Overall Progress"): # Use tqdm for overall progress

    print(f"\n{'='*60}\nDownloading data for year: {year}\n{'='*60}")

    single_file = os.path.join(SINGLE, f"{year}.nc")
    pressure_file = os.path.join(PRESSURE, f"{year}.nc")

    # ----------------------------
    # SINGLE LEVELS
    # ----------------------------
    single_level_variables = [
        "mean_sea_level_pressure",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "land_sea_mask"
    ]

    download_era5_data(
        dataset_name="reanalysis-era5-single-levels",
        variables=single_level_variables,
        pressure_level=None,
        year=year,
        output_file=single_file
    )

    # ----------------------------
    # PRESSURE LEVELS
    # ----------------------------
    pressure_level_variables = [
        "geopotential"
    ]
    pressure_levels = [
        "500"
    ]

    download_era5_data(
        dataset_name="reanalysis-era5-pressure-levels",
        variables=pressure_level_variables,
        pressure_level=pressure_levels,
        year=year,
        output_file=pressure_file
    )

print("\nFinished all downloads!")

### Session terminated so continuing ERA5 Data Download

Starting the ERA5 data download from 2012, ensuring existing files are not overwritten.

In [ ]:
#session terminated in between,so cotinued using:
import os
import cdsapi
import time
import calendar
from tqdm.auto import tqdm

# --------------------------------------------------
# SAVE LOCATION
# --------------------------------------------------

BASE = "/content/drive/MyDrive/isro_climate_dataset"

SINGLE = os.path.join(BASE, "ERA5", "SingleLevels")
PRESSURE = os.path.join(BASE, "ERA5", "PressureLevels")

os.makedirs(SINGLE, exist_ok=True)
os.makedirs(PRESSURE, exist_ok=True)

client = cdsapi.Client()

# Modified to start from 2012 as requested
YEARS = list(range(2012, 2026))
MONTHS = [f"{i:02d}" for i in range(1, 13)]

# Changed to 00:00 UTC as recommended
TIME = ["00:00"]

# India Bounding Box
AREA = [
    38.5,   # North
    68.0,   # West
    6.0,    # South
    98.5    # East
]

def download_era5_data(
    dataset_name,
    variables,
    pressure_level,
    year,
    output_file,
    max_retries=5,
    delay=5
):
    """Downloads ERA5 data with retry and resume support."""

    if os.path.exists(output_file):
        print(f"  ✅ File already exists: {output_file}. Skipping download.")
        return

    print(f"  ⬇️ Downloading {dataset_name} for {year}...")

    for attempt in range(max_retries):
        try:
            request_payload = {
                "product_type": "reanalysis",
                "variable": variables,
                "year": str(year),
                "month": MONTHS,
                "day": [], # Will be populated per month
                "time": TIME,
                "area": AREA,
                "data_format": "netcdf"
            }

            if pressure_level: # Only add if specific to pressure levels
                 request_payload["pressure_level"] = pressure_level

            # Generate correct days for each month for the current year
            all_days = []
            for month_num_str in MONTHS:
                month_num = int(month_num_str)
                num_days = calendar.monthrange(year, month_num)[1]
                all_days.extend([f"{d:02d}" for d in range(1, num_days + 1)])
            request_payload["day"] = sorted(list(set(all_days))) # Use set to remove duplicates, then sort

            client.retrieve(
                dataset_name,
                request_payload,
                output_file
            )
            print(f"  ✅ Successfully downloaded {os.path.basename(output_file)}.")
            return

        except Exception as e:
            print(f"  ❌ Download failed for {os.path.basename(output_file)} (Attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in {delay} seconds...")
                time.sleep(delay)
                delay *= 2 # Exponential backoff
            else:
                print(f"  🛑 Max retries reached for {os.path.basename(output_file)}. Skipping this file.")
                return # Exit function if max retries reached


print("Starting ERA5 data download...")

for year in tqdm(YEARS, desc="Overall Progress"): # Use tqdm for overall progress

    print(f"\n{'='*60}\nDownloading data for year: {year}\n{'='*60}")

    single_file = os.path.join(SINGLE, f"{year}.nc")
    pressure_file = os.path.join(PRESSURE, f"{year}.nc")

    # ----------------------------
    # SINGLE LEVELS
    # ----------------------------
    single_level_variables = [
        "mean_sea_level_pressure",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "land_sea_mask"
    ]

    download_era5_data(
        dataset_name="reanalysis-era5-single-levels",
        variables=single_level_variables,
        pressure_level=None,
        year=year,
        output_file=single_file
    )

    # ----------------------------
    # PRESSURE LEVELS
    # ----------------------------
    pressure_level_variables = [
        "geopotential"
    ]
    pressure_levels = [
        "500"
    ]

    download_era5_data(
        dataset_name="reanalysis-era5-pressure-levels",
        variables=pressure_level_variables,
        pressure_level=pressure_levels,
        year=year,
        output_file=pressure_file
    )

print("\nFinished all downloads!")

Starting ERA5 data download...


Overall Progress:   0%|          | 0/14 [00:00<?, ?it/s]


  ⬇️ Downloading reanalysis-era5-single-levels for 2012...


2026-06-30 14:01:34,625 INFO Request ID is e46d3af1-a0bd-45ed-bef0-8976cbfff025
INFO:ecmwf.datastores.legacy_client:Request ID is e46d3af1-a0bd-45ed-bef0-8976cbfff025
2026-06-30 14:01:34,820 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-06-30 14:01:50,632 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-06-30 14:01:58,430 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


7fc990fe2e3e78b443264edf1f34b1b2.nc:   0%|          | 0.00/35.0M [00:00<?, ?B/s]

...

2f575cfe540a220dbd8215314c0bdfd0.nc:   0%|          | 0.00/34.8M [00:00<?, ?B/s]

  ✅ Successfully downloaded 2025.nc.
  ⬇️ Downloading reanalysis-era5-pressure-levels for 2025...


2026-06-30 15:02:33,699 INFO Request ID is 13cc5f62-9dbb-435a-af4f-6ec92e32f15f
INFO:ecmwf.datastores.legacy_client:Request ID is 13cc5f62-9dbb-435a-af4f-6ec92e32f15f
2026-06-30 15:02:33,917 INFO status has been updated to accepted
INFO:ecmwf.datastores.legacy_client:status has been updated to accepted
2026-06-30 15:03:12,657 INFO status has been updated to running
INFO:ecmwf.datastores.legacy_client:status has been updated to running
2026-06-30 15:03:55,764 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


704ee318442ee10f5399c00093d27b69.nc:   0%|          | 0.00/7.59M [00:00<?, ?B/s]

  ✅ Successfully downloaded 2025.nc.

Finished all downloads!
